In [39]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from collections import deque
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple
import heapq
import itertools
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [41]:
class Problem(ABC):
    """Abstract base class for a search problem."""
    @abstractmethod
    def initial_state(self) -> Any:
        """Return the start state."""
        pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
        """Return True if state is a goal state."""
        pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
        """Return the legal actions available in the given state."""
        pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
        """Return the next state after applying action in state."""
        pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
        """Return the cost of applying action in state to reach next_state."""
        pass


In [43]:

dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)

    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost

In [44]:
def reconstruct_path(node: Node) -> List[Any]:
    """Return the list of states from the root node to this node."""
    path = []
    while node is not None:
        path.append(node.state)
        node = node.parent
    path.reverse()
    return path


def reconstruct_actions(node: Node) -> List[Any]:
    """Return the list of actions from the root node to this node."""
    actions = []
    while node is not None and node.parent is not None:
        actions.append(node.action)
        node = node.parent
    actions.reverse()
    return actions


def result_to_row(result: SearchResult) -> Dict[str, Any]:
    """Convert a SearchResult object into a row for a pandas DataFrame."""
    return {
        "Algorithm": result.algorithm,
        "Status": result.status,
        "Solution depth": result.solution_depth,
        "Solution cost": result.solution_cost,
        "Nodes expanded": result.nodes_expanded,
        "Max frontier": result.max_frontier_size,
        "Reached states": result.reached_count,
    }


def show_results(results: List[SearchResult]) -> pd.DataFrame:
    """Display results as a DataFrame."""
    return pd.DataFrame([result_to_row(r) for r in results])

In [45]:
MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}

class GridProblem(Problem):
    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        start, goal:
            Tuples in the form (row, col).
        """
        self.grid = grid
        self.start = start
        self.goal = goal

        self.rows = len(grid)
        self.cols = len(grid[0])

    def initial_state(self) -> Tuple[int, int]:
        return self.start

    def is_goal(self, state: Tuple[int, int]) -> bool:
        return state == self.goal

        raise NotImplementedError("Complete GridProblem.is_goal")

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_free(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return self.grid[row][col] == 0

    def actions(self, state: Tuple[int, int]) -> List[str]:
        legal_actions=[]

        for action in MOVES:
            row,col=state
            dr,dc= MOVES[action]
            new_state= (dr+row,dc+col)

            if self.in_bounds(new_state) and self.is_free(new_state):
                legal_actions.append(action)

        return legal_actions
        raise NotImplementedError("Complete GridProblem.actions")

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        row,col=state
        dr,dc = MOVES[action]
        return (row+dr,col+dc)

        raise NotImplementedError("Complete GridProblem.result")

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:
        return 1
        raise NotImplementedError("Complete GridProblem.action_cost")


class SearchAlgorithm(ABC):
    """Base class for search algorithms."""

    def expand(self, problem: Problem, node: Node) -> Iterable[Node]:
        state = node.state

        for action in problem.actions(state):
            next_state = problem.result(state,action)
            cost=(
                    node.path_cost + problem.action_cost
            (state,action,next_state)
            )

            yield Node(state = next_state,
                       parent = node,
                       action= action,
                       path_cost = cost
                       )
        raise NotImplementedError("Complete SearchAlgorithm.expand")

    @abstractmethod
    def search(self, problem: Problem) -> SearchResult:
        pass

In [46]:
def plot_path(
    grid: List[List[int]],
    start: Optional[Tuple[int, int]] = None,
    goal: Optional[Tuple[int, int]] = None,
    path: Optional[List[Tuple[int, int]]] = None,
    terrain_costs: Optional[List[List[float]]] = None,
    title: str = "Grid Map",
):
    """Visualise a grid and, optionally, a solution path (from Part A)."""
    arr = np.array(grid)
    height, width = arr.shape

    path_set = set(path) if path is not None else set()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

    for row in range(height):
        for col in range(width):
            state = (row, col)

            if arr[row, col] == 1:
                fill = (0.15, 0.15, 0.15)
            elif start is not None and state == start:
                fill = (0.95, 0.20, 0.20)
            elif goal is not None and state == goal:
                fill = (0.20, 0.70, 0.25)
            elif state in path_set:
                fill = (0.95, 0.90, 0.35)
            else:
                fill = (0.95, 0.95, 0.95)

            rect = patches.Rectangle(
                (col, row),
                1,
                1,
                linewidth=1,
                edgecolor=(0.75, 0.75, 0.75),
                facecolor=fill,
            )
            ax.add_patch(rect)

            if terrain_costs is not None and arr[row, col] == 0:
                ax.text(
                    col + 0.5,
                    row + 0.5,
                    str(terrain_costs[row][col]),
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    plt.show()

In [52]:
class WeightedGridProblem(GridProblem):
    """A grid problem where entering a cell costs that cell's terrain cost."""

    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
        terrain_costs: List[List[float]],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        terrain_costs:
            2D list, same shape as grid. terrain_costs[r][c] is the cost of
            ENTERING cell (r, c). All values must be >= 1.
        """
        super().__init__(grid, start, goal)
        self.terrain_costs = terrain_costs

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:

        row,col=next_state
        return self.terrain_costs[row][col]

        raise NotImplementedError("Complete WeightedGridProblem.action_cost")

In [53]:
wtest_grid = [
   [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
]

wtest_costs = [
    [1, 7, 1],
    [1, 1, 1],
    [1, 1, 1],
]

wtest_problem = WeightedGridProblem(
    wtest_grid, start=(0, 0), goal=(2, 2), terrain_costs=wtest_costs
)

# Entering the windy cell (0, 1) costs 7.
assert wtest_problem.action_cost((0, 0), "RIGHT", (0, 1)) == 7
# Entering the calm cell (1, 0) costs 1.
assert wtest_problem.action_cost((0, 0), "DOWN", (1, 0)) == 1
# Everything inherited from GridProblem still works.
assert wtest_problem.actions((0, 0)) == ["DOWN", "RIGHT"]
assert wtest_problem.is_goal((2, 2)) is True

print("WeightedGridProblem self-check passed.")


WeightedGridProblem self-check passed.


In [ ]:
def manhattan_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    r1,c1 = state
    r2,c2 = goal
    return abs(r1-r2) + abs(c1-c2)

    raise NotImplementedError("Complete manhattan_distance")


def euclidean_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    return math.dist(state,goal)

    raise NotImplementedError("Complete euclidean_distance")


def zero_heuristic(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    """h(n) = 0 for every node. Provided.

    A* with the zero heuristic degenerates into Uniform-Cost Search —
    admissible, consistent, and completely uninformative.
    """
    return 0.0

In [ ]:
assert manhattan_distance((3, 5), (3, 5)) == 0
assert abs(euclidean_distance((0, 0), (3, 4)) - 5.0) < 1e-9
assert euclidean_distance((1, 1), (1, 1)) == 0
assert zero_heuristic((0, 0), (9, 9)) == 0

# Euclidean never exceeds Manhattan (so Manhattan dominates Euclidean).
for s in [(0, 0), (2, 7), (5, 1)]:
    assert euclidean_distance(s, (9, 9)) <= manhattan_distance(s, (9, 9)) + 1e-9

print("Heuristic self-check passed.")

In [ ]:
class PriorityQueue:
    """A min-priority queue of (priority, node) pairs built on heapq."""

    def __init__(self):
        self._heap: List[Tuple[float, int, Node]] = []
        self._counter = itertools.count()

    def push(self, priority: float, node: Node) -> None:
        heapq.heappush(self._heap, (priority, next(self._counter), node))

    def pop(self) -> Node:
        """Remove and return the node with the LOWEST priority."""
        priority, count, node = heapq.heappop(self._heap)
        return node

    def __len__(self) -> int:
        return len(self._heap)

    def __bool__(self) -> bool:
        return len(self._heap) > 0

In [ ]:
class BestFirstSearch(SearchAlgorithm):
    """Generic best-first search. Subclasses define the evaluation function f(n)."""

    algorithm_name = "BestFirst"

    def __init__(
        self,
        heuristic: Callable[[Tuple[int, int], Tuple[int, int]], float] = zero_heuristic,
    ):
        self.heuristic = heuristic

    def h(self, node: Node, problem: Problem) -> float:
        """Heuristic estimate from this node's state to the goal."""
        return self.heuristic(node.state, problem.goal)

    def evaluation(self, node: Node, problem: Problem) -> float:
        """f(n). Subclasses override this single method."""

    raise NotImplementedError("Subclasses must define evaluation(node, problem)")

   def search(self, problem: Problem) -> SearchResult:

    node = Node(problem.initial_state())
    frontier = PriorityQueue()

    frontier.push(node, self.evaluation(node, problem)
    )

    reached = {
        node.state: node
    }

    nodes_expanded = 0
    max_frontier_size = 1

    while len(frontier) > 0:
        node = frontier.pop()

        if problem.is_goal(node.state):
            return SearchResult(
                algorithm=self.algorithm_name,
                status="success",
                solution=node,
                nodes_expanded=nodes_expanded,
                max_frontier_size=max_frontier_size,
                reached_count=len(reached)
            )
        nodes_expanded += 1

        for child in self.expand(problem, node):

            s = child.state

            if (
                s not in reached
                or child.path_cost < reached[s].path_cost
            ):

                reached[s] = child

                frontier.push(
                    child,
                    self.evaluation(child, problem)
                )

        max_frontier_size = max(
            max_frontier_size,
            len(frontier)
        )

    return SearchResult(
        algorithm=self.algorithm_name,
        status="failure",
        solution=None,
        nodes_expanded=nodes_expanded,
        max_frontier_size=max_frontier_size,
        reached_count=len(reached)
    )


    raise NotImplementedError("Complete BestFirstSearch.search")

In [ ]:
class GreedyBestFirstSearch(BestFirstSearch):
    algorithm_name = "Greedy"

    def evaluation(self, node: Node, problem: Problem) -> float:
        return self.h(node,problem)

        raise NotImplementedError("Complete GreedyBestFirstSearch.evaluation")

In [ ]:
class AStarSearch(BestFirstSearch):
    algorithm_name = "A*"

    def evaluation(self, node: Node, problem: Problem) -> float:
        return node.path_cost + self.h(node,problem)

        raise NotImplementedError("Complete AStarSearch.evaluation")

In [ ]:
class UniformCostSearch(BestFirstSearch):
    """UCS = best-first search with f(n) = g(n). Provided."""

    algorithm_name = "UCS"

    def __init__(self):
        super().__init__(heuristic=zero_heuristic)

    def evaluation(self, node: Node, problem: Problem) -> float:
        return node.path_cost

In [ ]:
class WeightedAStarSearch(BestFirstSearch):
    def __init__(self, heuristic, weight: float = 2.0):
        super().__init__(heuristic=heuristic)
        self.weight = weight
        self.algorithm_name = f"Weighted A* (W={weight})"

    def evaluation(self, node: Node, problem: Problem) -> float:
        return  node.path_cost + self.weight * self.h(node,problem)


        raise NotImplementedError("Complete WeightedAStarSearch.evaluation")

In [ ]:
check_grid = [
    [0, 0, 0],
    [1, 1, 0],
    [0, 0, 0],
]
check_problem = GridProblem(check_grid, start=(0, 0), goal=(2, 2))

for algo in [
    GreedyBestFirstSearch(manhattan_distance),
    AStarSearch(manhattan_distance),
    WeightedAStarSearch(manhattan_distance, weight=2),
    UniformCostSearch(),
]:
    res = algo.search(check_problem)
    assert res.status == "success", f"{res.algorithm} failed to find a path"
    assert res.solution_cost == 4, f"{res.algorithm} returned cost {res.solution_cost}, expected 4"
    assert res.path[0] == (0, 0) and res.path[-1] == (2, 2)

# A failure case: goal completely walled off.
walled = GridProblem(
    [[0, 1, 0],
     [1, 1, 0],
     [0, 0, 0]],
    start=(0, 0),
    goal=(2, 2),
)
assert AStarSearch(manhattan_distance).search(walled).status == "failure"

print("Algorithm self-check passed.")


In [ ]:
sample_grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
]

start = (0, 0)
goal = (9, 9)

uniform_problem = GridProblem(sample_grid, start, goal)

greedy = GreedyBestFirstSearch(manhattan_distance)
astar = AStarSearch(manhattan_distance)
wastar2 = WeightedAStarSearch(manhattan_distance, weight=2)
ucs = UniformCostSearch()

uniform_results = [
    ucs.search(uniform_problem),
    astar.search(uniform_problem),
    wastar2.search(uniform_problem),
    greedy.search(uniform_problem),
]

show_results(uniform_results)


In [ ]:
# Visualise two solution paths on the uniform map.
plot_path(
    sample_grid,
    start,
    goal,
    path=uniform_results[1].path,
    title="A* Solution Path (uniform costs)",
)

plot_path(
    sample_grid,
    start,
    goal,
    path=uniform_results[3].path,
    title="Greedy Best-First Solution Path (uniform costs)",
)


In [ ]:
turbulence_grid = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
]

turbulence_costs = [
    [1, 1, 1, 1, 1, 1, 1],
    [1, 1, 9, 9, 9, 1, 1],
    [1, 1, 9, 9, 9, 1, 1],
    [1, 1, 9, 9, 9, 1, 1],
    [1, 1, 1, 1, 1, 1, 1],
]

t_start = (2, 0)
t_goal = (2, 6)

turbulence_problem = WeightedGridProblem(
    turbulence_grid, t_start, t_goal, terrain_costs=turbulence_costs
)

turbulence_results = [
    ucs.search(turbulence_problem),
    astar.search(turbulence_problem),
    wastar2.search(turbulence_problem),
    greedy.search(turbulence_problem),
]

show_results(turbulence_results)

In [ ]:
# Greedy vs A* on the turbulence map. The numbers printed in each cell
# are the terrain costs.
plot_path(
    turbulence_grid,
    t_start,
    t_goal,
    path=turbulence_results[3].path,
    terrain_costs=turbulence_costs,
    title="Greedy: straight through the turbulence",
)

plot_path(
    turbulence_grid,
    t_start,
    t_goal,
    path=turbulence_results[1].path,
    terrain_costs=turbulence_costs,
    title="A*: routes around the turbulence",
)

In [ ]:
def inflated_heuristic(factor: float):
    def h(state, goal):
        return factor * manhattan_distance(state, goal)
    return h


    raise NotImplementedError("Complete inflated_heuristic")


# Once TODO 8 is complete, run the comparison.
#
# The "mild turbulence" map below is designed to expose the inflated
# heuristic: the direct route through the cost-3 band costs 12, while the
# calm detour costs 10. An overconfident heuristic charges straight ahead.

mild_grid = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
]

mild_costs = [
    [1, 1, 1, 1, 1, 1, 1],
    [1, 1, 3, 3, 3, 1, 1],
    [1, 1, 3, 3, 3, 1, 1],
    [1, 1, 3, 3, 3, 1, 1],
    [1, 1, 1, 1, 1, 1, 1],
]

mild_problem = WeightedGridProblem(
    mild_grid, (2, 0), (2, 6), terrain_costs=mild_costs
)

astar_honest = AStarSearch(manhattan_distance)
astar_inflated = AStarSearch(inflated_heuristic(3.0))

experiment_results = [
    astar_honest.search(uniform_problem),
    astar_inflated.search(uniform_problem),
    astar_honest.search(mild_problem),
    astar_inflated.search(mild_problem),
]

# Relabel rows so the table is readable.
experiment_results[0].algorithm = "A* admissible (uniform map)"
experiment_results[1].algorithm = "A* inflated x3 (uniform map)"
experiment_results[2].algorithm = "A* admissible (mild turbulence)"
experiment_results[3].algorithm = "A* inflated x3 (mild turbulence)"

show_results(experiment_results)

In [ ]:
# TODO 10:

custom_grid_1 = [
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0]
]

custom_costs_1 = [
    [1,1,1,1,1,1,1],
    [1,5,5,5,5,5,1],
    [1,5,5,5,5,5,1],
    [1,5,5,5,5,5,1],
    [1,1,1,1,1,1,1]
]
custom_start_1 = (2,0)
custom_goal_1 = (2,6)





In [ ]:
custom_problem_1 = WeightedGridProblem(
    custom_grid_1,
    custom_start_1,
    custom_goal_1,
    terrain_costs=custom_costs_1
)

In [ ]:
custom_results_1 = [
    ucs.search(custom_problem_1),
    astar.search(custom_problem_1),
    wastar2.search(custom_problem_1),
    greedy.search(custom_problem_1),
]
show_results(custom_results_1)

The map contains an expensive direct route and a cheaper detour. UCS and A* should find the cheapest path because they account for path cost. Greedy and Weighted A* may choose the more direct route because the heuristic strongly favors moving toward the goal, even when the terrain cost is higher.


In [ ]:
# TODO 11:

custom_grid_2 = [
    [0,0,0,0,0,0,0],
    [0,1,1,1,1,1,0],
    [0,1,0,0,0,1,0],
    [0,1,0,0,0,1,0],
    [0,1,1,1,0,1,0],
    [0,0,0,0,0,0,0]
]

custom_costs_2 = [
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1]
]

custom_start_2 = (2,2)
custom_goal_2 = (2,6)



In [ ]:
custom_problem_2 = WeightedGridProblem(
    custom_grid_2,
    custom_start_2,
    custom_goal_2,
    terrain_costs=custom_costs_2
)

In [ ]:
custom_results_2 = [
    ucs.search(custom_problem_2),
    astar.search(custom_problem_2),
    wastar2.search(custom_problem_2),
    greedy.search(custom_problem_2),
]
show_results(custom_results_2)

The U-shaped obstacle creates a heuristic trap. Greedy Best-First Search tends to move toward the goal and may spend time exploring inside the trap before backing out. A* performs better because it balances the path cost and heuristic estimate, while UCS remains optimal but usually expands more nodes.


1.
h(n) estimates the remaining cost from the current state to the goal. In this lab, its knowledge comes from the grid coordinates and the Manhattan distance calculation, which uses the current position and goal position to estimate how far apart they are.

 2.
The relaxation removes obstacles and terrain costs. It assumes the drone can always move directly toward the goal using horizontal and vertical moves without being blocked or affected by expensive terrain.

3.
Manhattan distance dominates Euclidean distance because it gives a larger estimate while still remaining admissible. A dominating heuristic is usually more informed, so A* expands fewer nodes when using Manhattan distance.

4.

If every move costs at least 1, the actual path cost can never be smaller than the Manhattan distance, so Manhattan distance remains admissible. If some terrain costs were 0.5, the true path cost could become smaller than the Manhattan estimate, causing the heuristic to overestimate and lose admissibility.



Greedy Best-First Search
1.
Greedy ignores g(n), the cost already accumulated. It only considers h(n), the estimated distance to the goal. On the turbulence map, Greedy preferred the direct route through expensive terrain because it looked closer to the goal, resulting in a more costly path.

2.

Expanding fewer nodes only means the search was faster. An algorithm must also consider solution quality. Greedy can find a path quickly but may return a much more expensive path than A*, so fewer expansions alone do not make it better.

3.
Greedy would be acceptable in an emergency search-and-rescue mission where finding a path quickly is more important than finding the cheapest path. A slightly more expensive route may be acceptable if it saves valuable time.


16.3 A* Search

1.
The drone evaluates a location by adding the cost already spent reaching that location to the estimated cost needed to reach the goal from there.

2.
A node may be generated through a path that is not optimal. Waiting until the node is popped ensures it has the lowest possible f-value and therefore represents the best path found so far.

3.
In BFS, all moves have equal cost, so the first visit to a state is always the best one. In A*, different paths can reach the same state with different costs, so the algorithm must remember the best node found for each state and replace worse ones when necessary.

4.
A* expanded fewer nodes than UCS because the heuristic guided the search toward the goal. This shows that a good heuristic can significantly reduce search effort while still finding the optimal solution.


16.4 Admissibility and Consistency

1.

A heuristic is admissible if it never overestimates the true remaining cost to the goal. A heuristic is consistent if, for every move, the estimated cost does not decrease by more than the cost of the move. Consistency implies admissibility.

 2.

The experiment showed that A* is only guaranteed to find the optimal path when the heuristic is admissible. After inflating the heuristic, A* expanded fewer nodes but sometimes returned a more expensive path.

3.

Yes. Moving one step changes the Manhattan distance by at most 1, which is equal to the step cost. Therefore h(n) ≤ c(n,a,n') + h(n'), satisfying the consistency condition.

---

 16.5 Weighted A* and Trade-offs

1.

When W = 0, Weighted A* behaves like UCS because only g(n) matters. When W = 1, it becomes standard A*. As W becomes very large, the heuristic dominates and the behaviour approaches Greedy Best-First Search.

2.

Weighted A* with weight W guarantees a solution cost no more than W times the optimal cost when the heuristic is admissible. In my experiments, the returned costs stayed within this bound.

3.

I would choose Weighted A* with a moderate weight such as W = 2. It reduces computation time and node expansions while usually producing solutions close to optimal, making it a good balance between speed and path quality.


16.6 Memory and Real-World Drone Context

1.
A* stores a frontier and a reached dictionary, which can grow very large. IDA* avoids this by using depth-first style searches with limited memory, but it pays by re-expanding many nodes and increasing computation time.

2.
Terrain costs could include wind strength, battery consumption, restricted airspace, weather conditions, obstacle density, and risk levels associated with different regions.

 3.
The heuristic may become inaccurate because the goal position changes while the search is running. The planned path may no longer lead to the target, requiring continuous replanning and updated heuristics.

4.
I would deploy A* because it provides an excellent balance between efficiency and optimality. I would reconsider using Weighted A* if decisions had to be made much faster or if hardware limitations made standard A* too slow.